# Preparación de datos (módulo 01)
Este cuaderno deja listos los datos para todo el proyecto: inventariamos las 9075 imágenes de SylFishBD, agrupamos los duplicados, hacemos la partición train/val/test por grupos, generamos las máscaras de segmentación con OpenCV y guardamos las cachés que usarán la CNN y la U-Net. Todo lo que producimos aquí (`splits.csv`, máscaras y archivos `.npz`) se guarda en `MyDrive/sylfish/`.

Podemos ejecutarlo completo de arriba abajo: cada paso largo revisa primero si su resultado ya está en Drive (y es consistente) y, de ser así, lo carga en vez de recalcularlo. Para forzar un paso basta poner su entrada en `True` en el diccionario `REHACER` de la configuración.

In [ ]:
"""
Autor: Jairo de Jesús Blanco Pérez
Codigo: (completar)
Materia: Inteligencia Artificial
Tema: Módulo 01 - Preparación de datos (particiones por grupo, máscaras y cachés)
"""
# imports
import os # manejo, recorrido y listado de carpetas
import json # guardar parámetros
import time # medir tiempos
import zipfile # descomprimir dataset
import hashlib # huellas MD5
import random # numeros aleatorios
import shutil # copiar y mover archivos
from pathlib import Path # manejo de rutas
from concurrent.futures import ThreadPoolExecutor # paralelismo para las máscaras
import numpy as np # manejo de arrays y operaciones matematicas
import pandas as pd # tablas
import cv2 # OpenCV para manejo de imagenes
import matplotlib.pyplot as plt # graficas
from tqdm import tqdm # barras de progreso
from scipy.sparse import coo_matrix # matriz dispersa para el grafo de duplicados
from scipy.sparse.csgraph import connected_components # componentes conexas
from sklearn.model_selection import StratifiedGroupKFold # partición por grupos, no por imagenes
from google.colab import drive # conexion entre colab y drive

## **Configuración**
Fijamos en una sola celda la semilla, las rutas, los tamaños y los parámetros del módulo. La primera parte es idéntica en los cuatro cuadernos del proyecto; la segunda es propia de este módulo.

In [ ]:
SEED = 42  # semilla para que la distribución de datos sea la misma en todo el proyecto
random.seed(SEED)
np.random.seed(SEED)

# rutas en Drive (persisten)
DRIVE_ROOT = Path("/content/drive/MyDrive/sylfish")
ZIP_DRIVE = DRIVE_ROOT / "data" / "raw" / "SylFishBD_IMAGE.zip"
SPLITS_DIR = DRIVE_ROOT / "data" / "splits"
PROCESSED_DIR = DRIVE_ROOT / "data" / "processed"
MASKS_ZIP = DRIVE_ROOT / "data" / "masks.zip"
TABLES_DIR = DRIVE_ROOT / "reports" / "tables"
FIGURES_DIR = DRIVE_ROOT / "reports" / "figures"

# rutas locales (se borran al reiniciar la sesión)
LOCAL_ROOT = Path("/content")
EXTRACT_DIR = LOCAL_ROOT / "sylfishbd"
MASKS_DIR = LOCAL_ROOT / "masks"

# clases en el orden de classes.txt (el índice es el label): 0 - boal, 1 - ilish, ..., 8 - telapia
CLASES = ["boal", "ilish", "kalibaush", "katla", "koi",
          "mrigel", "pabda", "rui", "telapia"]
NUM_CLASES = len(CLASES)
CLASE_A_LABEL = {}
for i, c in enumerate(CLASES):
    CLASE_A_LABEL[c] = i

# tamaños de imagen e interpolaciones para bajar carga computacional
IMG_CNN = 224
IMG_UNET = 256
INTER_IMG = cv2.INTER_AREA        # imágenes: promedia píxeles al reducir
INTER_MASK = cv2.INTER_NEAREST    # máscaras: no inventa grises

# partición por grupos: 20 folds -> 14 train (0.70), 3 val (0.15), 3 test (0.15)
N_FOLDS = 20
FOLDS_TEST = [0, 1, 2]
FOLDS_VAL = [3, 4, 5]

# umbral por defecto de la máscara de la U-Net (se ajusta en val)
UMBRAL_MASCARA = 0.5

# ---------- propio del módulo 01 ----------
N_IMAGENES = 9075                 # imágenes esperadas en el dataset
UMBRAL_HAMMING = 5                # distancia de Hamming máxima para considerar casi duplicadas
MD5_SPLITS_PREFIJO = "5b174a51"   # MD5 de splits.csv que ya se le comunicó al equipo

# máscaras (GrabCut) y control de calidad
PARAMS_MASK = {"lado": 196, "margen": 0.05, "iteraciones": 8, "k_apertura": 5}
QC = {"area_min": 0.02, "area_max": 0.70, "rect_max": 0.30}   # area_max se decide en D5

# True fuerza a rehacer ese paso aunque ya exista su resultado en Drive
REHACER = {"inventario": False, "pares": False, "splits": False,
           "mascaras": False,  # OJO: regenerar sobrescribe masks.zip, que es la referencia entregada
           "cache_cnn": False, "cache_unet": False}

# archivos de salida
INVENTARIO_CSV = SPLITS_DIR / "inventario.csv"
PARES_CSV = SPLITS_DIR / "pares_duplicados.csv"
SPLITS_CSV = SPLITS_DIR / "splits.csv"
EXCLUIDAS_CSV = SPLITS_DIR / "excluidas.csv"
QC_CSV = PROCESSED_DIR / "mask_qc.csv"
PARAMS_JSON = PROCESSED_DIR / "mask_params.json"

## **A. Montaje y datos**

Montamos Drive, comprobamos que el zip del dataset está y creamos las carpetas de salida (si ya existen, no pasa nada).

In [ ]:
drive.mount('/content/drive')
assert ZIP_DRIVE.exists(), f"no encontramos el zip en {ZIP_DRIVE}"

for carpeta in [SPLITS_DIR, PROCESSED_DIR, TABLES_DIR, FIGURES_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)
print("Drive listo")

Ahora definimos dos funciones auxiliares de guardado. Escriben primero a un archivo temporal y lo renombran al terminar, así una desconexión de Colab no deja un archivo a medias que luego parezca válido.

In [ ]:
def guardar_atomico(local, destino):
    # copia un archivo local a Drive con nombre temporal y lo renombra al final
    tmp = destino.with_name(destino.name + ".tmp")
    shutil.copy(local, tmp)
    tmp.replace(destino)

def guardar_csv(tabla, destino, index=False):
    # guarda una tabla en csv con la misma lógica atómica
    tmp = destino.with_name(destino.name + ".tmp")
    tabla.to_csv(tmp, index=index)
    tmp.replace(destino)

Luego pasamos el zip a local. Solo lo descomprimimos si en `/content` no están ya las 9075 imágenes.

In [ ]:
n_jpg = len(list(EXTRACT_DIR.rglob("*.jpg"))) if EXTRACT_DIR.exists() else 0
if n_jpg != N_IMAGENES:
    shutil.rmtree(EXTRACT_DIR, ignore_errors=True)  # descartamos una extracción incompleta
    zip_local = LOCAL_ROOT / ZIP_DRIVE.name
    shutil.copy(ZIP_DRIVE, zip_local)
    with zipfile.ZipFile(zip_local) as z:
        z.extractall(EXTRACT_DIR)
    zip_local.unlink()  # liberamos disco
    print("descomprimido en", EXTRACT_DIR)
else:
    print("ya existe", EXTRACT_DIR)

Guardamos la ruta de las imágenes (la carpeta `images` que contiene las 9 clases), definimos la lectura en RGB y contamos imágenes por clase.

In [ ]:
candidatos = []
for p in EXTRACT_DIR.rglob("images"):
    if p.is_dir() and all((p / c).is_dir() for c in CLASES):
        candidatos.append(p)
IMAGES_DIR = candidatos[0]

def leer_rgb(ruta):
    # ruta relativa a images/ -> imagen RGB (OpenCV lee en BGR)
    img = cv2.imread(str(IMAGES_DIR / ruta))
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

total = 0
for c in CLASES:
    n_c = len(list((IMAGES_DIR / c).glob("*.jpg")))
    print(f"{c:10s} jpg: {n_c:5d}")
    total += n_c
print("total jpg:", total)
assert total == N_IMAGENES, f"esperábamos {N_IMAGENES} imágenes"

## **B. Inventario y duplicados**
Armamos el dataframe base: cada fila es una imagen con su ruta relativa (`filepath`), su clase (`clase`) y su etiqueta numérica (`label`, de 0 a 8). A cada imagen le calculamos además su MD5, que detecta copias exactas, y el hash perceptual dHash de $8 \times 8$ (también de su versión volteada), que detecta fotos casi idénticas. Guardamos todo en `inventario.csv` para no recalcularlo.

In [ ]:
filas = []
for c in CLASES:
    for ruta in sorted((IMAGES_DIR / c).glob("*.jpg")):
        filas.append({"filepath": f"{c}/{ruta.name}",
                      "clase": c,
                      "label": CLASE_A_LABEL[c]})
df = pd.DataFrame(filas)
print(df.shape)
print(df["clase"].value_counts())

Definimos las dos funciones de huella: `md5_archivo` (contenido exacto del archivo) y `dhash` (compara cada píxel de una versión reducida con su vecino de la derecha y guarda los 64 bits como texto hexadecimal).

In [ ]:
def md5_archivo(ruta):
    # huella del contenido exacto del archivo (copias exactas)
    return hashlib.md5(ruta.read_bytes()).hexdigest()

def dhash(gris, tam=8):
    # reducir a (tam+1) x tam y comparar cada píxel con su vecino de la derecha
    chico = cv2.resize(gris, (tam + 1, tam), interpolation=cv2.INTER_AREA)
    dif = chico[:, 1:] > chico[:, :-1]
    bits = 0
    for b in dif.flatten():
        bits = (bits << 1) | int(b)
    return format(bits, "016x")   # 64 bits como texto hexadecimal (casi duplicados)

Ahora leemos todas las imágenes de una sola pasada y guardamos el inventario. Si `inventario.csv` ya existe y corresponde a este dataset, lo cargamos. Los hashes se leen como texto para no perder los ceros a la izquierda.

In [ ]:
df_inv = None
if INVENTARIO_CSV.exists() and not REHACER["inventario"]:
    tmp = pd.read_csv(INVENTARIO_CSV, dtype={"md5": str, "dhash": str, "dhash_flip": str})
    if tmp["filepath"].tolist() == df["filepath"].tolist():
        df_inv = tmp
        print("inventario cargado de Drive")

if df_inv is None:
    registros = []
    for fp in tqdm(df["filepath"]):
        ruta = IMAGES_DIR / fp
        img = cv2.imread(str(ruta))          # BGR, o None si está dañada
        if img is None:
            registros.append({"leida": False})
            continue
        alto, ancho, canales = img.shape
        gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        registros.append({"leida": True,
                          "alto": alto, "ancho": ancho, "canales": canales,
                          "bytes": ruta.stat().st_size,
                          "md5": md5_archivo(ruta),
                          "dhash": dhash(gris),
                          "dhash_flip": dhash(cv2.flip(gris, 1))})   # imagen volteada
    df_inv = pd.concat([df, pd.DataFrame(registros)], axis=1)
    guardar_csv(df_inv, INVENTARIO_CSV)
    print("inventario calculado y guardado")

df = df_inv
print(df.shape)

Hacemos las consultas de consistencia: imágenes que no se leen, tamaños distintos, copias exactas y copias exactas con clases distintas (ruido de etiqueta).

In [ ]:
print("no leídas:", (~df["leida"]).sum())
print(df.groupby(["ancho", "alto", "canales"]).size())      # tamaños distintos
print("MD5 repetidos:", df["md5"].duplicated().sum())
n_clases = df.groupby("md5")["clase"].nunique()
print("MD5 en más de una clase:", (n_clases > 1).sum())

md5_mixto = n_clases[n_clases > 1].index   # las que pertenecen a varias clases
print(df[df["md5"].isin(md5_mixto)].sort_values("md5")[["filepath", "md5"]])

Ahora buscamos los casi duplicados: para cada imagen calculamos la distancia de Hamming $d_H$ entre su dHash y el de todas las siguientes, comparando también contra la versión volteada (copias en espejo), y guardamos los pares con $d_H \le 5$ en `pares_duplicados.csv`. Si ese archivo ya existe y coincide con el inventario, lo cargamos.

In [ ]:
def hamming_uno_contra_todos(valor, arreglo):
    # XOR deja en 1 los bits distintos; se cuentan con unpackbits
    xor = arreglo ^ valor
    bits = np.unpackbits(xor.view(np.uint8).reshape(-1, 8), axis=1)
    return bits.sum(axis=1)

pares_df = None
if PARES_CSV.exists() and not REHACER["pares"]:
    tmp = pd.read_csv(PARES_CSV)
    if len(tmp) == 0 or (tmp["j"].max() < len(df)
                         and (df["filepath"].values[tmp["i"]] == tmp["fp_i"].values).all()
                         and (df["filepath"].values[tmp["j"]] == tmp["fp_j"].values).all()):
        pares_df = tmp
        print("pares cargados de Drive")

if pares_df is None:
    h = np.array([int(x, 16) for x in df["dhash"]], dtype=np.uint64)
    hf = np.array([int(x, 16) for x in df["dhash_flip"]], dtype=np.uint64)
    pares = []
    for i in tqdm(range(len(df) - 1)):
        d_normal = hamming_uno_contra_todos(h[i], h[i + 1:])
        d_flip = hamming_uno_contra_todos(hf[i], h[i + 1:])   # copia en espejo
        d = np.minimum(d_normal, d_flip)
        for k in np.where(d <= UMBRAL_HAMMING)[0]:
            pares.append((i, i + 1 + k, int(d[k])))
    pares_df = pd.DataFrame(pares, columns=["i", "j", "dist"])
    pares_df["fp_i"] = df["filepath"].values[pares_df["i"]]
    pares_df["fp_j"] = df["filepath"].values[pares_df["j"]]
    pares_df["clase_i"] = df["clase"].values[pares_df["i"]]
    pares_df["clase_j"] = df["clase"].values[pares_df["j"]]
    guardar_csv(pares_df, PARES_CSV)   # se filtra por dist después
    print("pares calculados y guardados")

print("pares:", len(pares_df))
print(pares_df["dist"].value_counts().sort_index())
print("pares entre clases distintas:", (pares_df["clase_i"] != pares_df["clase_j"]).sum())

Revisamos visualmente algunos pares: los más lejanos (distancia 4 o 5), que son los más dudosos, y los pares entre clases distintas que no son copias exactas.

In [ ]:
def ver_pares(tabla):
    fig, ejes = plt.subplots(len(tabla), 2, figsize=(6, 3 * len(tabla)), squeeze=False)
    for fila, (_, p) in enumerate(tabla.iterrows()):
        for col, fp in enumerate([p["fp_i"], p["fp_j"]]):
            ejes[fila, col].imshow(leer_rgb(fp))
            ejes[fila, col].set_title(f"{fp} (d={p['dist']})", fontsize=7)
            ejes[fila, col].axis("off")
    plt.tight_layout()
    plt.show()

lejanos = pares_df[pares_df["dist"] >= 4]
if len(lejanos) > 0:
    ver_pares(lejanos.sample(min(6, len(lejanos)), random_state=SEED))

cruzados = pares_df[pares_df["clase_i"] != pares_df["clase_j"]]   # pares entre clases distintas
print(cruzados["dist"].value_counts().sort_index())
print(cruzados.groupby(["clase_i", "clase_j"]).size())
no_exactos = cruzados[cruzados["dist"] > 0]
if len(no_exactos) > 0:
    ver_pares(no_exactos.sample(min(6, len(no_exactos)), random_state=SEED))

### Grupos de duplicados
Pensamos cada imagen como un nodo y unimos con una arista a las que forman un par casi duplicado. Cada componente conexa del grafo es un grupo de imágenes que provienen de la misma foto, y cada imagen recibe un `group_id`. Así la partición se hace por grupo y no por imagen, y una misma foto no puede quedar en dos conjuntos distintos.

La función `agrupar` recibe un umbral, filtra los pares, arma la matriz de adyacencia y devuelve una etiqueta de grupo por imagen. Comparamos los umbrales 5 y 3 con el número de grupos, el tamaño del grupo más grande y cuántos grupos mezclan más de una clase: un grupo gigante indica un umbral muy permisivo, y un grupo con varias clases indica ruido de etiqueta o falsos positivos.

In [ ]:
def agrupar(umbral):
    p = pares_df[pares_df["dist"] <= umbral]  # pares que consideramos duplicados
    A = coo_matrix((np.ones(len(p)), (p["i"], p["j"])), shape=(len(df), len(df)))  # adyacencia
    _, etiquetas = connected_components(A, directed=False)  # una etiqueta por imagen
    return etiquetas

for umbral in (5, 3):
    g = df.assign(group_id=agrupar(umbral)).groupby("group_id")
    tam = g.size()  # imágenes por grupo
    ncl = g["clase"].nunique()  # clases por grupo
    print(f"d_H <= {umbral}: grupos={len(tam)}, tamaño máx={tam.max()}, "
          f"multiclase={(ncl > 1).sum()}, grupos de 1 imagen={(tam == 1).sum()}")

## **C. Particiones**
### Exclusión de grupos conflictivos
Con el umbral $d_H \le 5$ asignamos el `group_id`. Excluimos los grupos que contienen más de una clase, porque no sabemos cuál etiqueta es la correcta, y el bloque `rui_791` a `rui_810`, que son copias de imágenes de `mrigel`. Reportamos cuántas imágenes se pierden.

In [ ]:
df["group_id"] = agrupar(UMBRAL_HAMMING)  # con 5 se minimizan las inconsistencias según las pruebas

ncl = df.groupby("group_id")["clase"].transform("nunique")  # clases por grupo, por fila
bloque_rui = {f"rui/rui_{k}.jpg" for k in range(791, 811)}  # bloque sospechoso
g_bloque = set(df.loc[df["filepath"].isin(bloque_rui), "group_id"])  # sus grupos completos

excluir = (ncl > 1) | df["group_id"].isin(g_bloque)  # grupos multiclase o del bloque
print("multiclase:", (ncl > 1).sum(), "| bloque rui:", df["group_id"].isin(g_bloque).sum(), "| excluidas:", excluir.sum())
df_ok = df[~excluir].reset_index(drop=True)  # dataset limpio
print("imágenes restantes:", len(df_ok))

### Partición por grupos
Dividimos en entrenamiento, validación y prueba (70/15/15) con `StratifiedGroupKFold` de 20 folds: 3 folds para test, 3 para val y 14 para train. La partición es por grupo y estratificada por clase. Al final verificamos las proporciones y que ningún grupo aparezca en más de un conjunto.

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)  # folds de ~5 %
nombres = np.empty(len(df_ok), dtype=object)
for k, (_, idx_fold) in enumerate(sgkf.split(df_ok, df_ok["label"], df_ok["group_id"])):
    if k in FOLDS_TEST:
        nombres[idx_fold] = "test"
    elif k in FOLDS_VAL:
        nombres[idx_fold] = "val"
    else:
        nombres[idx_fold] = "train"
df_ok["split"] = nombres

print(df_ok["split"].value_counts(normalize=True).round(3))  # debe rondar 0.70 / 0.15 / 0.15
print(pd.crosstab(df_ok["clase"], df_ok["split"], normalize="index").round(3))  # proporción por clase
print("máx splits por grupo:", df_ok.groupby("group_id")["split"].nunique().max())  # debe ser 1

### Guardado de `splits.csv`
`splits.csv` es el contrato de todo el equipo, así que si ya existe en Drive **no lo sobrescribimos**: lo cargamos y comprobamos que la partición recién calculada es idéntica. Si no lo es (por ejemplo, otra versión de scikit-learn), avisamos y seguimos con el archivo de Drive. También guardamos `excluidas.csv` y mostramos el MD5 de `splits.csv`.

In [ ]:
cols = ["filepath", "clase", "label", "group_id", "split"]
splits_nuevo = df_ok[cols].reset_index(drop=True)

if SPLITS_CSV.exists() and not REHACER["splits"]:
    igual = hashlib.md5(SPLITS_CSV.read_bytes()).hexdigest() == hashlib.md5(splits_nuevo.to_csv(index=False).encode()).hexdigest()
    print("splits.csv ya existe en Drive;", "la partición recalculada es idéntica" if igual else "ATENCIÓN: la recalculada es distinta, usamos la de Drive")
    splits_df = pd.read_csv(SPLITS_CSV)
else:
    guardar_csv(splits_nuevo, SPLITS_CSV)
    splits_df = splits_nuevo
    print("splits.csv guardado")

excluidas_df = df[~df["filepath"].isin(splits_df["filepath"])][["filepath", "clase", "group_id"]]
guardar_csv(excluidas_df, EXCLUIDAS_CSV)   # para reportarlas en el informe

md5_splits = hashlib.md5(SPLITS_CSV.read_bytes()).hexdigest()   # para avisarle al equipo
print(len(splits_df), len(excluidas_df), md5_splits)
if not md5_splits.startswith(MD5_SPLITS_PREFIJO):
    print("ATENCIÓN: el MD5 no coincide con el que ya se comunicó al equipo")
print(splits_df["split"].value_counts())
print(excluidas_df["clase"].value_counts())  # excluidas por clase
print(splits_df["filepath"].iloc[0])         # debe ser relativa a images/

## **D. Máscaras de segmentación**
SylFishBD no trae máscaras y la U-Net necesita una por imagen: pez $= 255$, fondo $= 0$. Las generamos con **GrabCut**, un algoritmo clásico de OpenCV que aprende el color del fondo con lo que queda fuera de un rectángulo central y separa el pez con un corte de grafo.

### D1. Funciones
Definimos `generar_mascara` (GrabCut en baja resolución, limpieza con apertura, componente conexa mayor, relleno de huecos y regreso a 500×500 binarizado en $\{0, 255\}$) y `metricas_qc` (fracción de la imagen que es pez y fracción de una banda de 5 px justo dentro del rectángulo de GrabCut). GrabCut inicializa con k-means aleatorio, por eso fijamos la semilla en cada llamada.

In [ ]:
def generar_mascara(img_rgb, lado, margen, iteraciones, k_apertura):
    cv2.setRNGSeed(SEED)  # k-means de GrabCut reproducible (el generador es propio de cada hilo)
    h, w = img_rgb.shape[:2]
    chica = cv2.resize(img_rgb, (lado, lado), interpolation=cv2.INTER_AREA)  # GrabCut en baja resolución
    m = int(margen * lado)
    rect = (m, m, lado - 2 * m, lado - 2 * m)           # (x, y, ancho, alto); lo de afuera es fondo seguro
    mask_gc = np.zeros((lado, lado), np.uint8)
    bgd = np.zeros((1, 65), np.float64)                  # modelos internos de fondo y primer plano
    fgd = np.zeros((1, 65), np.float64)
    cv2.grabCut(chica, mask_gc, rect, bgd, fgd, iteraciones, cv2.GC_INIT_WITH_RECT)
    fg = np.where((mask_gc == cv2.GC_FGD) | (mask_gc == cv2.GC_PR_FGD), 255, 0).astype(np.uint8)
    # limpieza: apertura, componente mayor y relleno de huecos
    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_apertura, k_apertura))
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, ker)
    n, etiquetas, stats, _ = cv2.connectedComponentsWithStats(fg, connectivity=8)
    if n <= 1:
        return np.zeros((h, w), np.uint8), 0             # máscara vacía
    mayor = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])   # la etiqueta 0 es el fondo
    comp = np.where(etiquetas == mayor, 255, 0).astype(np.uint8)
    contornos, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    final = np.zeros_like(comp)
    cv2.drawContours(final, contornos, -1, 255, thickness=-1)
    # volvemos a 500x500 suavizando el borde y re-binarizando a {0, 255}
    final = cv2.resize(final, (w, h), interpolation=cv2.INTER_LINEAR)
    final = np.where(final > 127, 255, 0).astype(np.uint8)
    return final, n - 1

def metricas_qc(mask, margen):
    h, w = mask.shape
    m, b = int(margen * w), 5                            # banda de 5 px justo dentro del rectángulo de GrabCut
    banda = np.zeros(mask.shape, dtype=bool)
    banda[m:m + b, m:w - m] = True; banda[h - m - b:h - m, m:w - m] = True
    banda[m:h - m, m:m + b] = True; banda[m:h - m, w - m - b:w - m] = True
    return (mask > 0).mean(), (mask[banda] > 0).mean()   # fracción de pez y fracción de banda ocupada

### D2. Generación (o carga) de las máscaras
Generamos las 8975 máscaras con dos hilos y las guardamos como PNG $\{0, 255\}$ a resolución original; para cada una registramos el área y la fracción de banda ocupada. Usamos hilos y no procesos porque OpenCV junto con `fork` se colgaba, y GrabCut corre solo en CPU, así que no hace falta GPU.

La celda decide sola qué hacer:
1. Si faltan máscaras en `/content` y existe `masks.zip` en Drive, lo descomprime.
2. Si están todas y `mask_qc.csv` está completo, no calcula nada.
3. Si falta alguna, genera solo las que faltan.
4. Si generó alguna nueva, actualiza `masks.zip` en Drive.

Las máscaras de `masks.zip` son la referencia entregada: la primera pasada se hizo antes de fijar la semilla de GrabCut, por lo que regenerarlas desde cero puede dar diferencias pequeñas.

In [ ]:
cv2.setNumThreads(1)  # cada hilo de Python usa un solo hilo de OpenCV

def procesar(ruta):
    destino = MASKS_DIR / Path(ruta).with_suffix(".png")
    mask = cv2.imread(str(destino), cv2.IMREAD_GRAYSCALE) if destino.exists() else None  # reanuda si ya existe
    generada = mask is None                              # no existía o quedó corrupta por la interrupción
    if generada:
        mask, _ = generar_mascara(leer_rgb(ruta), **PARAMS_MASK)
        destino.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(destino), mask)                  # PNG {0, 255} a 500x500
    area_frac, rect_frac = metricas_qc(mask, PARAMS_MASK["margen"])
    return ruta, area_frac, rect_frac, generada

def n_pngs():
    return sum(1 for _ in MASKS_DIR.rglob("*.png")) if MASKS_DIR.exists() else 0

rutas = splits_df["filepath"].tolist()

if REHACER["mascaras"]:  # empezamos de cero (ignora masks.zip y el QC guardado)
    shutil.rmtree(MASKS_DIR, ignore_errors=True)
    QC_CSV.unlink(missing_ok=True)

# 1) si faltan máscaras locales, las traemos de Drive
if n_pngs() < len(rutas) and MASKS_ZIP.exists() and not REHACER["mascaras"]:
    zip_local = LOCAL_ROOT / "masks_drive.zip"
    shutil.copy(MASKS_ZIP, zip_local)
    with zipfile.ZipFile(zip_local) as z:
        z.extractall(MASKS_DIR)
    zip_local.unlink()
    print("máscaras descomprimidas desde Drive:", n_pngs())

# 2) si ya tenemos las métricas de todas, no recalculamos
crudo = None
if n_pngs() >= len(rutas) and QC_CSV.exists():
    tmp = pd.read_csv(QC_CSV)
    if set(tmp["filepath"]) == set(rutas):
        crudo = tmp[["filepath", "area_frac", "rect_frac"]]
        print("métricas cargadas de Drive")

# 3) si no, generamos las que falten (y medimos todas)
nuevas = 0
if crudo is None:
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=2) as pool:
        resultados = list(tqdm(pool.map(procesar, rutas), total=len(rutas)))
    print(f"tiempo total: {(time.time() - t0) / 60:.1f} min")
    crudo = pd.DataFrame([r[:3] for r in resultados], columns=["filepath", "area_frac", "rect_frac"])
    nuevas = sum(r[3] for r in resultados)
    print("máscaras generadas ahora:", nuevas)

assert n_pngs() == len(rutas), f"faltan máscaras: {n_pngs()} de {len(rutas)}"

# 4) si hay máscaras nuevas (o no hay zip en Drive), actualizamos masks.zip
if nuevas > 0 or not MASKS_ZIP.exists():
    shutil.make_archive("/content/masks", "zip", MASKS_DIR)   # zip local
    guardar_atomico(Path("/content/masks.zip"), MASKS_ZIP)
    print("masks.zip actualizado en Drive:", round(MASKS_ZIP.stat().st_size / 1e6, 1), "MB")

### D3. Control de calidad
Marcamos como sospechosas las máscaras vacías, con área muy chica o muy grande, o que llenan la banda interior del rectángulo (pez cortado o máscara desbordada). Los umbrales están en `QC` de la configuración. Esta celda es barata y se ejecuta siempre: así, si cambiamos un umbral, `mask_qc.csv` y `mask_params.json` se actualizan sin regenerar nada.

In [ ]:
def motivo(fila):
    m = []
    if fila.area_frac == 0:
        m.append("vacia")
    elif fila.area_frac < QC["area_min"]:
        m.append("area_chica")
    if fila.area_frac > QC["area_max"]:
        m.append("area_grande")
    if fila.rect_frac > QC["rect_max"]:
        m.append("llena_rect")
    return "|".join(m)

qc_df = crudo.copy()
qc_df["motivo"] = [motivo(f) for f in qc_df.itertuples()]
qc_df["qc_ok"] = qc_df["motivo"] == ""
guardar_csv(qc_df, QC_CSV)
with open(PARAMS_JSON, "w") as f:
    json.dump({**PARAMS_MASK, **QC, "metodo": "grabcut"}, f, indent=2)   # parámetros de generación y de QC

print(qc_df["qc_ok"].value_counts())
print(qc_df["motivo"].value_counts())
print(qc_df[["area_frac", "rect_frac"]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

q = qc_df.merge(splits_df[["filepath", "clase", "split"]], on="filepath")
print(q.groupby("split")["qc_ok"].agg(["sum", "count", "mean"]))                      # impacto por partición
print(q.pivot_table(index="clase", columns="split", values="qc_ok", aggfunc="sum"))  # válidas por clase y partición

### D4. Inspección visual
Definimos una función que dibuja el contorno de la máscara guardada sobre la imagen, y la usamos para mirar las marcadas por el QC. Las vacías no se pueden recuperar con técnicas clásicas: en muchas el fondo es muy parecido al pez en color y textura, así que en la U-Net las omitimos (sesgo documentado en el informe) y las conservamos en la caché de la CNN.

In [ ]:
def grilla_mascaras(rutas, titulos, ncols=6):
    filas = (len(rutas) + ncols - 1) // ncols
    fig, ejes = plt.subplots(filas, ncols, figsize=(3 * ncols, 3.2 * filas), squeeze=False)
    for eje in ejes.ravel():
        eje.axis("off")
    for eje, ruta, t in zip(ejes.ravel(), rutas, titulos):
        mask = cv2.imread(str(MASKS_DIR / Path(ruta).with_suffix(".png")), cv2.IMREAD_GRAYSCALE)  # máscara guardada
        contornos, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        sobre = leer_rgb(ruta)
        cv2.drawContours(sobre, contornos, -1, (255, 0, 0), 3)
        eje.imshow(sobre); eje.set_title(f"{Path(ruta).name} {t}", fontsize=7)
    plt.tight_layout()
    plt.show()

def muestra_area(tabla, a, b, n=12):
    sub = tabla[(tabla["area_frac"] >= a) & (tabla["area_frac"] < b)]
    return sub.sample(min(n, len(sub)), random_state=SEED) if len(sub) > 0 else sub

vacias = qc_df[qc_df["motivo"] == "vacia"]["filepath"]
print(len(vacias), "vacías; por clase:", vacias.str.split("/").str[0].value_counts().to_dict())

marcadas = qc_df[(~qc_df["qc_ok"]) & (qc_df["area_frac"] > 0)]   # sospechosas con máscara no vacía
print(len(marcadas), "marcadas no vacías")
if len(marcadas) > 0:
    m8 = marcadas.sample(min(8, len(marcadas)), random_state=SEED)
    grilla_mascaras(m8["filepath"].tolist(), m8["motivo"].tolist(), ncols=4)

### D5. Calibración de los umbrales de área
Miramos solo las máscaras no vacías (las vacías ya son fallos seguros). Para el **área mínima** contamos cuántas quedan marcadas con cada valor candidato y vemos los tramos $[0.01, 0.02)$ y $[0.02, 0.03)$: por debajo de $0.02$ son fragmentos de aleta, y por encima, peces pequeños bien segmentados. Para el **área máxima** medimos el efecto de bajarla sobre las imágenes válidas por clase y partición, y miramos muestras por rango de área.

In [ ]:
no_vacias = qc_df[qc_df["area_frac"] > 0]

print("area mínima: máscaras no vacías por debajo de cada candidato")
for t in [0.01, 0.02, 0.03, 0.05, 0.08]:
    print("  0 < area <", t, (no_vacias["area_frac"] < t).sum())

for a, b in [(0.01, 0.02), (0.02, 0.03)]:
    s = muestra_area(no_vacias, a, b, n=8)
    if len(s) > 0:
        print(f"tramo {a} a {b}")
        grilla_mascaras(s["filepath"].tolist(), [f"{x:.3f}" for x in s["area_frac"]], ncols=4)

In [ ]:
# efecto de area_max sobre las válidas por clase y partición (solo medimos; no cambia QC)
q = qc_df.merge(splits_df[["filepath", "clase", "split"]], on="filepath")
candidatos_max = [0.30, 0.35, 0.45, 0.70]
tabla_max = {}
for amax in candidatos_max:
    valida = (q["area_frac"] >= QC["area_min"]) & (q["area_frac"] <= amax) & (q["rect_frac"] <= QC["rect_max"])
    tabla_max[amax] = valida
    print(f"area_max={amax:.2f}: válidas={valida.sum()} ({valida.mean():.1%})")

resumen_max = pd.DataFrame({f"{amax:.2f}": tabla_max[amax].groupby([q["clase"], q["split"]]).sum() for amax in candidatos_max}).unstack("split")
print(resumen_max)   # filas = clase; columnas = (area_max, partición)

In [ ]:
ok = qc_df[qc_df["qc_ok"]]
for a, b in [(0.02, 0.15), (0.15, 0.30), (0.30, 0.45), (0.45, 0.70)]:
    print(f"area {a} a {b}: {((ok['area_frac'] >= a) & (ok['area_frac'] < b)).sum()} imágenes válidas")
    s = muestra_area(ok, a, b, n=12)
    if len(s) > 0:
        grilla_mascaras(s["filepath"].tolist(), [f"{x:.2f}" for x in s["area_frac"]])

## **E. Cachés para los modelos**
Las cachés guardan las imágenes ya reducidas en un `.npz` por partición, para que los cuadernos de entrenamiento no lean 9000 JPG cada vez. Antes de reconstruir una caché comprobamos que existe, que sus `filepath` y etiquetas coinciden con `splits.csv` y que tres imágenes recalculadas desde el JPG son idénticas a las guardadas; si algo falla, se rehace y avisa.

In [ ]:
def cache_valida(path, sub, claves, calcular):
    # True si el .npz coincide con la partición y 3 muestras recalculadas son idénticas
    if not path.exists():
        return False
    try:
        d = np.load(path, allow_pickle=True)
        if not np.array_equal(d["filepath"], sub["filepath"].to_numpy()):
            return False
        if not np.array_equal(d["y"], sub["label"].to_numpy(dtype=np.int64)):
            return False
        arrs = {c: d[c] for c in claves}                       # cargamos una sola vez
        for k in np.linspace(0, len(sub) - 1, 3).astype(int):
            esperado = calcular(sub["filepath"].iloc[k])
            for c in claves:
                if not np.array_equal(arrs[c][k], esperado[c]):
                    return False
        return True
    except Exception as e:
        print("  caché ilegible:", e)
        return False

### E1. Caché de la CNN
Leemos cada imagen, la pasamos de BGR a RGB y la reducimos a $224 \times 224$ con `INTER_AREA`. Quedan guardadas `X` (uint8), `y` y `filepath`. Incluye todas las imágenes de `splits.csv`, también las de máscara vacía.

In [ ]:
def calcular_cnn(ruta):
    return {"X": cv2.resize(leer_rgb(ruta), (IMG_CNN, IMG_CNN), interpolation=INTER_IMG)}

for split in ["train", "val", "test"]:
    sub = splits_df[splits_df["split"] == split].reset_index(drop=True)
    destino = PROCESSED_DIR / f"cnn_224_{split}.npz"
    if not REHACER["cache_cnn"] and cache_valida(destino, sub, ["X"], calcular_cnn):
        print(split, "caché CNN al día, se omite")
        continue
    X = np.zeros((len(sub), IMG_CNN, IMG_CNN, 3), dtype=np.uint8)  # reservamos memoria
    for k, ruta in enumerate(tqdm(sub["filepath"], desc=split)):
        X[k] = calcular_cnn(ruta)["X"]
    y = sub["label"].to_numpy(dtype=np.int64)
    fp = sub["filepath"].to_numpy()
    local = LOCAL_ROOT / f"cnn_224_{split}.npz"                # guardamos local y luego copiamos a Drive
    np.savez_compressed(local, X=X, y=y, filepath=fp)
    guardar_atomico(local, destino)
    print(split, X.shape, X.dtype, y.shape, np.bincount(y, minlength=NUM_CLASES))

In [ ]:
datos = np.load(PROCESSED_DIR / "cnn_224_train.npz", allow_pickle=True)   # recargamos desde Drive
X_cnn, fp_cnn = datos["X"], datos["filepath"]
idxs = np.linspace(0, len(X_cnn) - 1, 8).astype(int)                       # ocho índices repartidos
fig, ejes = plt.subplots(1, 8, figsize=(24, 3.4))
for eje, i in zip(ejes, idxs):
    eje.imshow(X_cnn[i]); eje.set_title(fp_cnn[i], fontsize=7); eje.axis("off")   # colores e imagen deben coincidir con el nombre
plt.show()
del X_cnn

### E2. Caché de la U-Net
Reducimos cada imagen a $256 \times 256$ con `INTER_AREA` y su máscara con `INTER_NEAREST`, que no inventa valores intermedios y deja la máscara solo en $\{0, 255\}$. Guardamos `X`, `M`, `y` y `filepath` por partición. La caché incluye todas las imágenes; el filtro `qc_ok` se aplica en el cuaderno 03.

In [ ]:
def calcular_unet(ruta):
    m = cv2.imread(str(MASKS_DIR / Path(ruta).with_suffix(".png")), cv2.IMREAD_GRAYSCALE)
    return {"X": cv2.resize(leer_rgb(ruta), (IMG_UNET, IMG_UNET), interpolation=INTER_IMG),
            "M": cv2.resize(m, (IMG_UNET, IMG_UNET), interpolation=INTER_MASK)}

for split in ["train", "val", "test"]:
    sub = splits_df[splits_df["split"] == split].reset_index(drop=True)
    destino = PROCESSED_DIR / f"unet_256_{split}.npz"
    if not REHACER["cache_unet"] and cache_valida(destino, sub, ["X", "M"], calcular_unet):
        print(split, "caché U-Net al día, se omite")
        continue
    X = np.zeros((len(sub), IMG_UNET, IMG_UNET, 3), dtype=np.uint8)  # imágenes
    M = np.zeros((len(sub), IMG_UNET, IMG_UNET), dtype=np.uint8)     # máscaras
    for k, ruta in enumerate(tqdm(sub["filepath"], desc=split)):
        par = calcular_unet(ruta)
        X[k], M[k] = par["X"], par["M"]
    y = sub["label"].to_numpy(dtype=np.int64)
    fp = sub["filepath"].to_numpy()
    local = LOCAL_ROOT / f"unet_256_{split}.npz"
    np.savez_compressed(local, X=X, M=M, y=y, filepath=fp)
    guardar_atomico(local, destino)
    print(split, X.shape, M.shape, np.unique(M), round((M > 0).mean(), 3))  # M solo {0, 255}; fracción media de pez

In [ ]:
d = np.load(PROCESSED_DIR / "unet_256_val.npz", allow_pickle=True)   # recargamos desde Drive
X, M, fp = d["X"], d["M"], d["filepath"]
idxs = np.linspace(0, len(X) - 1, 8).astype(int)                      # ocho índices repartidos
fig, ejes = plt.subplots(2, 8, figsize=(24, 6))
for k, i in enumerate(idxs):
    ejes[0, k].imshow(X[i]); ejes[0, k].set_title(fp[i], fontsize=7); ejes[0, k].axis("off")
    ejes[1, k].imshow(M[i], cmap="gray"); ejes[1, k].axis("off")      # la máscara debe coincidir con el pez de arriba
plt.show()
del X, M

## **F. Cierre**
Guardamos en `reports/` las tablas y figuras que usa el informe (se sobrescriben en cada ejecución, son baratas) y verificamos que todos los productos del módulo están en Drive.

In [ ]:
# tablas
guardar_csv(pd.crosstab(splits_df["clase"], splits_df["split"]), TABLES_DIR / "01_conteo_clase_split.csv", index=True)
guardar_csv(excluidas_df["clase"].value_counts().rename_axis("clase").reset_index(name="excluidas"), TABLES_DIR / "01_excluidas_por_clase.csv")
q = qc_df.merge(splits_df[["filepath", "clase", "split"]], on="filepath")
validas = q.pivot_table(index="clase", columns="split", values="qc_ok", aggfunc="sum")
guardar_csv(validas, TABLES_DIR / "01_qc_validas_clase_split.csv", index=True)
guardar_csv(qc_df["motivo"].replace("", "ok").value_counts().rename_axis("motivo").reset_index(name="n"), TABLES_DIR / "01_qc_motivos.csv")
guardar_csv(resumen_max, TABLES_DIR / "01_efecto_area_max.csv", index=True)

# figuras
fig, eje = plt.subplots(figsize=(6, 3.5))
eje.hist(qc_df.loc[qc_df["area_frac"] > 0, "area_frac"], bins=60, color="steelblue")
eje.axvline(QC["area_min"], color="red", ls="--", label=f"area_min = {QC['area_min']}")
eje.axvline(QC["area_max"], color="darkred", ls="--", label=f"area_max = {QC['area_max']}")
eje.set_xlabel("fracción de la imagen que es pez"); eje.set_ylabel("imágenes"); eje.legend()
fig.tight_layout(); fig.savefig(FIGURES_DIR / "01_qc_hist_area.png", dpi=200); plt.show()

ax = validas[["train", "val", "test"]].plot(kind="bar", figsize=(7, 3.5))
ax.set_ylabel("imágenes con máscara válida"); ax.set_xlabel("")
plt.tight_layout(); plt.savefig(FIGURES_DIR / "01_qc_validas_clase_split.png", dpi=200); plt.show()

# verificación final
esperados = [SPLITS_CSV, EXCLUIDAS_CSV, INVENTARIO_CSV, PARES_CSV, QC_CSV, PARAMS_JSON, MASKS_ZIP]
for s in ["train", "val", "test"]:
    esperados += [PROCESSED_DIR / f"cnn_224_{s}.npz", PROCESSED_DIR / f"unet_256_{s}.npz"]
for ruta in esperados:
    print(("ok " if ruta.exists() else "FALTA "), ruta.relative_to(DRIVE_ROOT), f"{ruta.stat().st_size / 1e6:.1f} MB" if ruta.exists() else "")
print("MD5 splits.csv:", hashlib.md5(SPLITS_CSV.read_bytes()).hexdigest())